# PixelCNN vs Tiny LDM

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from gen_cats.evaluation.report_analysis import (
    PRIOR_DIR,
    ensure_plots_dir,
    load_prior_summary,
    prior_seed_assets,
    write_snippet,
    write_stats_csv,
)

sns.set_theme(style="whitegrid", context="talk")
PLOTS = ensure_plots_dir("results")
summary = load_prior_summary()
per_seed = pd.DataFrame(summary["per_seed"])
per_seed

,seed,vqvae_seed,num_embeddings,feature_map_size,recon_loss,n_samples,pixelcnn_checkpoint,tiny_ldm_checkpoint,pixelcnn_seconds,tiny_ldm_seconds,pixelcnn_seconds_per_image,tiny_ldm_seconds_per_image,speedup_ldm_over_pixelcnn
0,42,42,512,16,mse,16,checkpoints/pixelcnn/e8b7a925fdaf/best_seed42.pt,checkpoints/tiny_ldm/4968b94b662b/best_seed42.pt,4.161595,0.830253,0.260100,0.051891,5.012442
1,0,0,512,16,mse,16,checkpoints/pixelcnn/32a45e7e8ab6/best_seed0.pt,checkpoints/tiny_ldm/77b49ba7c148/best_seed0.pt,4.187360,0.823687,0.261710,0.051480,5.083679
2,3407,3407,512,16,mse,16,checkpoints/pixelcnn/099fc9ce2263/best_seed340...,checkpoints/tiny_ldm/e8e3810e09ea/best_seed340...,4.153190,0.765152,0.259574,0.047822,5.427928


In [2]:
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(per_seed))
width = 0.35
ax.bar([i - width / 2 for i in x], per_seed["pixelcnn_seconds_per_image"], width, label="PixelCNN")
ax.bar([i + width / 2 for i in x], per_seed["tiny_ldm_seconds_per_image"], width, label="Tiny LDM")
ax.set_xticks(list(x))
ax.set_xticklabels([f"seed {s}" for s in per_seed["seed"]])
ax.set_ylabel("seconds / image")
ax.legend()
fig.tight_layout()
out = PLOTS / "prior_timing.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.close(fig)
summary["mean_speedup_ldm_over_pixelcnn"]

5.174682974936221

In [3]:
from pathlib import Path

import matplotlib.image as mpimg
import numpy as np

SAMPLES_PER_ROW = 4
PRIOR_MODELS = [
    ("pixelcnn", "PixelCNN"),
    ("tiny_ldm", "Tiny LDM"),
]


def split_sample_grid(path: Path, grid: int = 4) -> list[np.ndarray]:
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    crop = (min(h, w) // grid) * grid
    off_y = (h - crop) // 2
    off_x = (w - crop) // 2
    img = img[off_y : off_y + crop, off_x : off_x + crop]
    tile_h, tile_w = crop // grid, crop // grid
    tiles: list[np.ndarray] = []
    for row in range(grid):
        for col in range(grid):
            tiles.append(
                img[row * tile_h : (row + 1) * tile_h, col * tile_w : (col + 1) * tile_w]
            )
    return tiles


def stitch_horizontal(parts: list[np.ndarray], gap: int) -> np.ndarray:
    if not parts:
        raise ValueError("stitch_horizontal requires at least one image")
    tile_h, tile_w = parts[0].shape[:2]
    channels = parts[0].shape[2] if parts[0].ndim == 3 else 1
    total_w = len(parts) * tile_w + (len(parts) - 1) * gap
    canvas = np.ones((tile_h, total_w, channels), dtype=parts[0].dtype)
    x0 = 0
    for part in parts:
        canvas[:, x0 : x0 + tile_w] = part
        x0 += tile_w + gap
    return canvas


def stitch_model_row(model_key: str, seeds: list[int], *, tile_gap: int = 2, group_gap: int = 12) -> np.ndarray:
    groups = [
        stitch_horizontal(split_sample_grid(prior_seed_assets(seed, PRIOR_DIR)[model_key])[:SAMPLES_PER_ROW], tile_gap)
        for seed in seeds
    ]
    return stitch_horizontal(groups, group_gap)


seeds = summary["seeds"]
row_images = [stitch_model_row(model_key, seeds) for model_key, _ in PRIOR_MODELS]
fig_width = 10.5
row_height = fig_width / (row_images[0].shape[1] / row_images[0].shape[0])
title_space = 0.24
seed_header = 0.18
fig_height = len(row_images) * (row_height + title_space) + seed_header + 0.08

fig, axes = plt.subplots(len(row_images), 1, figsize=(fig_width, fig_height), squeeze=False)
for ax, (model_key, model_label), row_img in zip(axes[:, 0], PRIOR_MODELS, row_images, strict=True):
    ax.imshow(row_img, aspect="auto")
    ax.set_title(model_label, loc="left", fontsize=11, pad=5)
    ax.axis("off")

seed_ax = fig.add_axes([0.01, 0.94, 0.98, 0.05])
seed_ax.axis("off")
group_width = 1.0 / len(seeds)
for seed_i, seed in enumerate(seeds):
    seed_ax.text(
        (seed_i + 0.5) * group_width,
        0.5,
        f"seed {seed}",
        ha="center",
        va="center",
        fontsize=11,
        transform=seed_ax.transAxes,
    )

fig.subplots_adjust(hspace=0.42, left=0.002, right=0.998, top=0.90, bottom=0.002)
out = PLOTS / "prior_samples_panel.png"
fig.savefig(out, dpi=200, bbox_inches="tight", pad_inches=0.01)
plt.close(fig)
out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/prior_samples_panel.png')

In [4]:
pcnn_s = summary["mean_pixelcnn_seconds_per_image"]
ldm_s = summary["mean_tiny_ldm_seconds_per_image"]
speedup = summary["mean_speedup_ldm_over_pixelcnn"]
latex = f"""\\begin{{table}}[H]
    \\centering
    \\caption{{Wall-clock generation time per image (mean over seeds).}}
    \\label{{tab:prior_timing}}
    \\begin{{tabular}}{{lrr}}
        \\toprule
        PixelCNN & {pcnn_s:.3f} & 1.00$\\times$ \\\\
        Tiny LDM & {ldm_s:.3f} & {speedup:.2f}$\\times$ \\\\
        \\bottomrule
    \\end{{tabular}}
\\end{{table}}"""
write_snippet("prior_timing.tex", latex)
write_stats_csv(per_seed, "prior_timing.csv")

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/report_snippets/prior_timing.csv')